# 14 层次聚类 Hierarchical Clustering

依赖安装说明：`pip install numpy matplotlib scikit-learn`

层次聚类会构造样本之间的层级关系。凝聚式层次聚类从每个点都是一个簇开始，不断合并最近的簇。


## 0. 学习目标和阅读地图

层次聚类的重点是“先建立层级，再决定切几类”。你需要掌握：

1. 凝聚式聚类如何从点合并成簇。
2. linkage 选择如何影响簇形状。
3. 层次结构和最终簇标签的区别。
4. 它为什么适合小中型数据和探索分析。


## 1. 数学逻辑

凝聚式聚类流程：

1. 初始时每个样本都是一个簇。
2. 计算簇之间距离。
3. 合并距离最近的两个簇。
4. 重复直到达到目标簇数。

常见 linkage：

- single：两个簇中最近点的距离。
- complete：两个簇中最远点的距离。
- average：所有点对距离平均。
- ward：合并后簇内方差增加最小。


## 1.1 推导拆开看：簇之间距离不是唯一的

层次聚类的关键问题是：两个簇之间距离怎么算？

single linkage：

$$d(A,B)=\min_{a\in A,b\in B}d(a,b)$$

complete linkage：

$$d(A,B)=\max_{a\in A,b\in B}d(a,b)$$

average linkage：

$$d(A,B)=\frac{1}{|A||B|}\sum_{a\in A}\sum_{b\in B}d(a,b)$$

不同定义会产生不同层级结构。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs
from sklearn.cluster import AgglomerativeClustering

np.random.seed(42)
X, _ = make_blobs(n_samples=90, centers=3, cluster_std=0.75, random_state=42)


## 1.2 层次聚类的输出怎么理解

`AgglomerativeClustering` 最终给出每个样本的簇标签。但在概念上，它先构造了一棵合并树。

你可以把 `n_clusters=3` 理解为：在这棵合并树上切一刀，留下 3 个大分支。


In [ ]:
# 从零实现：非常小规模的 average linkage 合并过程

def cluster_distance(X, c1, c2):
    distances = []
    for i in c1:
        for j in c2:
            distances.append(np.sqrt(np.sum((X[i] - X[j]) ** 2)))
    return np.mean(distances)

clusters = [[i] for i in range(12)]  # 只用前 12 个点演示，避免输出太长
X_small = X[:12]
while len(clusters) > 3:
    best = None
    for i in range(len(clusters)):
        for j in range(i + 1, len(clusters)):
            dist = cluster_distance(X_small, clusters[i], clusters[j])
            if best is None or dist < best[0]:
                best = (dist, i, j)
    dist, i, j = best
    clusters[i] = clusters[i] + clusters[j]
    clusters.pop(j)
    print(f'合并后簇数量={len(clusters):2d} | 本次距离={dist:.3f}')

print('最终小样本簇:', clusters)


## 1.3 从零实现代码怎么读

为了避免输出过长，从零版本只取前 12 个点演示。每一轮：

1. 遍历所有簇对。
2. 用 average linkage 计算簇间距离。
3. 合并最近的两个簇。
4. 重复直到剩 3 个簇。

完整算法只是把这个过程应用到所有样本。


In [ ]:
for linkage in ['ward', 'complete', 'average', 'single']:
    model = AgglomerativeClustering(n_clusters=3, linkage=linkage)
    labels = model.fit_predict(X)
    plt.figure()
    plt.scatter(X[:,0], X[:,1], c=labels, cmap='tab10', s=28)
    plt.title(f'AgglomerativeClustering linkage={linkage}')
    plt.show()


In [ ]:
# 诊断：不同 linkage 产生的标签可能差异很大
for linkage in ['ward', 'complete', 'average', 'single']:
    lab = AgglomerativeClustering(n_clusters=3, linkage=linkage).fit_predict(X)
    counts = np.bincount(lab)
    print(f'{linkage:>8s} cluster sizes:', counts.tolist())


## 2.1 如何诊断层次聚类

层次聚类没有像监督学习那样的标准准确率。常见做法是：

- 对比不同 linkage 的稳定性。
- 看簇大小是否极端不平衡。
- 结合业务含义解释每个簇。
- 小数据可以画 dendrogram，大数据通常不方便。


## 2. 常见误区

- 层次聚类计算成本较高，大数据集上不一定合适。
- linkage 选择会显著影响结果。
- 它能给出层级结构，但最终切成几类仍然需要决策。

## 3. 小实验

- 改 `n_clusters`。
- 对比不同 linkage。
- 增大样本数，感受运行时间变化。


## 5. 复习清单

- 凝聚式聚类从每个点一个簇开始合并。
- linkage 定义簇间距离。
- 最终簇数是从层级结构里切出来的。
- 适合探索结构，但大数据上成本较高。
